In [1]:
# import everything first:
import numpy as np
import astropy.constants as const
import astropy.units as u
from matplotlib import pyplot as plt

In [2]:
# Constants
G = 6.674e-8        # cm^3 g^-1 s^-2
sigmaB = 5.67e-5    # erg cm^-2 s^-1 K^-4
kB = 1.381e-16      # erg/K
L_sun = 3.828e33    # erg/s
R_p = 7.149e9       # Jupiter radius in cm
AU = 1.496e13       # cm
L_jup = 3.846e30  # erg/s

# Physical constants
M_sun = 1.989e33 
M_jup = 1.898e30
R_jup = 7.149e9
#AU = 1.496e13
sec_per_yr = 3.156e7
Rgas = 8.314e7  # erg/(mol K)


In [3]:
# define a function for Lp and Rp based on mass 


In [23]:
def B_nu(nu, T):
    """
    Planck function in frequency (erg s^-1 cm^-2 Hz^-1 sr^-1)
    nu: frequency in Hz
    T: temperature in K
    """
    h = 6.626e-27  # erg*s
    c = 2.998e10   # cm/s
    k = 1.381e-16  # erg/K
    
    return (2*h*nu**3 / c**2) / (np.exp(h*nu/(k*T)) - 1)

def calculate_disk_properties_Andrews(M_star = 1, Mp = 1, Mdot = 1.56, alpha = 1e-3, kappaR = 10 , Rin = 1, rp = 22, d_pc = 167, 
                             lam_nu=240 , 
                             Lplanet=1e-6, Lstar = 1.0 , show_results = False, Rout=None, Rp = 1, M_cpd = 0.01, i = 16):
    """
    Calculate disk temperature profiles and millimeter flux.
    
    Parameters:
    -----------
    M_star : float
        Stellar mass (M_sun)
    lam_nu : float
        Wavelength (GHz)
    Mp : float
        Planet mass (M_jup)
    Mdot : float  
        Accretion rate (M_jup/Myr)
    alpha : float
        Viscosity parameter
    kappaR : float
        Rosseland opacity (cm^2/g)
    Rin : float
        Inner radius (Rjup)
    Rp : float
        distance of planet from star (au)  
    d_pc : float
        Distance (pc)
    mode : str
        Irradiation mode: "b", "no_b", or "planet"
    lam_mm : float
        Wavelength (mm)
    T_ISM : float
        Interstellar medium temperature (K)
    Lplanet : float
        Planet luminosity (L_sun)
    

    Lstar : float
        Stellar luminosity (L_sun)
        
    Returns:
    --------
    dict : Dictionary containing R, temperatures, Sigma, tau_mm, Tb, F_nu_tot
    """
    # Turn input values to cgs
    # Disk parameters
    M_star = M_star * M_sun
    Mp = Mp * M_jup
    Mdot = Mdot * M_jup / (sec_per_yr*1e6)
    Rin = Rin * R_jup
    rp = rp * AU # planet orbital radius in cm
    Rp = Rp * R_jup  # planet radius in cm

    # Turn the luminosites to erg/s
    Lplanet = Lplanet * L_sun
    Lstar = Lstar * L_sun
    
    # 1. Create radial grid between Rin and Rout

    

    if Rout is None:
        Rout = (1/3) * rp * (Mp / (3 * M_star))**(1/3)
    else:
        Rout = Rout * AU  # assume input Rout is in AU


    R = np.geomspace(Rin, Rout, 100)  # cm
    
    # 2. Calculate T(R)
    # Accretion irradiation
    
    T_irr = (3*G * Mp * Mdot / (8*np.pi*sigmaB*R**3) * (1 - np.sqrt(Rp/R)))**(1/4)  # K # RJUP CAN BE WRONG  NEED PLANET RADIUS
    
    #print(f'L_irr = {L_irr/L_sun:.2e} L_sun')

    # Planet irradiation

    T_irr_p = (0.1 * Lplanet  / (4 * np.pi * sigmaB * R**2))**0.25  # K  , R is coordinate in CPD frame

    # Star irradiation

    phi_flare = 0.02   # flaring angle of the host disk   (Huang et al. (2018b))

    T_irr_star = (phi_flare * Lstar / (8 * np.pi * sigmaB * rp**2))**0.25  # K

    T_ext= (T_irr**4 + T_irr_p**4 + T_irr_star**4 )**0.25  # K
    
    
    #---------------------------------------
    # -------------SIGMA--------------------
    # --------------------------------------
    
    gamma = 0.75
    M_cpd = M_cpd * M_jup  # CPD mass in grams


    Sigma0 = (2 - gamma) * M_cpd /  (2 * np.pi * Rout**gamma) * (Rout**(2 - gamma) - Rp**(2 - gamma))**(-1)
    Sigma = Sigma0 * (R/Rout)**(-gamma)




    #---------------------------------------
    #---------------Flux--------------------
    # --------------------------------------

    kappa_nu = 2.4 #cm^2/g at 1.3mm 

    tau_nu = kappa_nu * Sigma

    # Assume face-on disk: mu = cos(i) = 1

    # turn i from degree to radian
   
    mu = np.cos(np.radians(i))# i is the inclination angle in degree in table 2 

    # Convert distance to cm
    pc_to_cm = 3.086e18  # cm/pc
    d_cm = d_pc * pc_to_cm

    # Integrand: B_nu(T) * (1 - exp(-tau/mu)) * r
    # Ignoring omega_nu (scattering) term as specified
    integrand = B_nu(lam_nu * 1e9, T_ext) * (1 - np.exp(-tau_nu / mu)) * R

    # Integrate using trapezoidal rule
    flux_integral = np.trapz(integrand, R)

    # Total observed flux at Earth
    F_cpd = (2 * np.pi * mu / d_cm**2) * flux_integral  # erg s^-1 cm^-2 Hz^-1

    # Convert to microJy
    F_microJy = F_cpd / 1e-29  # µJy

    
    return {
        'R': R,
        'Rout': Rout / AU,
        'R_AU': R / AU,
        'T_ext': T_ext,
        'Sigma': Sigma,
        'tau_nu_max': np.max(tau_nu),
        'F_nu_tot': F_microJy,
        'M_cpd': M_cpd / M_jup,
        'T_irr': T_irr,
        'T_irr_p': T_irr_p,
        'T_irr_star': T_irr_star
    }




### Check model consistency with Andrews HD143006

Measure the flux limit at the innergap of the disk and try to obtain the same planet properties. Relevant to table 3 and table 4  - try to reproduce table 4.
since the observation is made in different wavelengths, I will take the flux he measured at the wavelength and the location of the gap, plug into my model to plot the radius versus planet mass for D22 and D51 gap. 

1. Change the gap location unit from mas to au (au = mas *dc/1000)
2. Makesure flux is in value of microJy (take the range of flux values for each recovery fraction to verify as it would make it more convenient for me if I want to do CPD injection recovery too) 
3. Plot them


*Note: the observing frequency is 240 GHz which is 1.25 mm , the wavelength-dependent opacity is 2.4 cm^2/g*


Okay , so the situation now is that , the contour levels should be changed into recovery fractions (if present) or sigma levels , the one overlaying line should be the ALMA angular resolution,  representing the locus of CPD parameters where the CPD angular size equals the beam: theta_cpd = Rcpd/d = theta_ALMA, therefore y axis (Rcpd/RHill) = theta_beam*d/RHill , where RHill depends on Mp. so thats the contour line that shows below which the CPD is unresolved. 
without the recovery fractions...what is the minimum flux? should I keep the whole recovery fraction relevant relation as an if case? 




In [30]:




def Andrews_plots_flux_map(target_flux_arr, disk_arr=None, rp=165, alpha=1e-3, plotmass=True , color = '5sigsource', y_max = 0.8, y_min= 0.05):
    """
    Make a DSHARP-style 2D flux map: 
    - x-axis: log10(Mp/Mjup)
    - y-axis: Rcpd / RHill -> normalised disk outer radius
    - shading: log10(Fnu [µJy]) or log10(M_dust [M_earth])
    - contours: recovery flux thresholds

    Parameters
    ----------
    target_flux_arr : list
        Flux thresholds (µJy) for contour level.
    disk_arr : list
        [disk_name, M_star (Msun), distance (pc), Lstar (Lsun)]
    rp : float
        Planet orbital radius (AU).
    lam_mm : float
        Observing wavelength in mm.
    alpha : float
        Viscosity parameter.
    """
    if disk_arr is None:
        disk_arr = ["Unknown", 1.0, 100, 1.0]

    # Grids
    Mp_grid = np.logspace(-2, 1.5, 100)       # 0.01 - 10 Mjup
    M_cpd_grid = np.logspace(-5.5,-3, 80 )
    Rout_frac_grid = np.linspace(y_min, y_max, 80)  # fraction of RH   -> w
    M_star_jup = disk_arr[1] * 1047.56     # convert Msun to Mjup for explicit RHill calc

    # Array of flux with 2D shape ( Rout_frac, Mp)
    Flux_vals = np.zeros((len(Rout_frac_grid), len(Mp_grid)))
    M_cpd_chosen = np.zeros((len(Rout_frac_grid), len(Mp_grid)))
    min_flux = target_flux_arr[0]  # minimum flux threshold

    # Compute flux grid, for value combination in Rout and Mp
    for i, frac in enumerate(Rout_frac_grid):
        for j, Mp in enumerate(Mp_grid):
            RHill = rp * (Mp / (3 * M_star_jup))**(1/3)
            Rout = frac * RHill
            
            # Add a simple Lplanet - Mp relation from 
            a = ((10**(-4)-10**(-6))/(3-0.4))
            b = ((10**(-6)-(10**(-8))/(0.4-0.025)))
            c = ((10**(-4)-(10**(-8))/(3-0.025)))

            slope = np.average([a,b,c])
            Lp = slope * Mp # in Lsun
            # Initialize
            chosen_flux = 0.0
            chosen_M_cpd = 0.0
            tau_max_chosen = 0.0
            prev_flux = 0.0
            flux_plateau_threshold = 0.05  # 5% change threshold
            
            # Loop through M_cpd from smallest to largest
            for M_cpd in M_cpd_grid:
                out = calculate_disk_properties_Andrews(
                    M_star=disk_arr[1], Mp=Mp, alpha=alpha,
                    Rin=1, rp=rp, Rp=1, d_pc=disk_arr[2],
                    lam_nu=240,  Lplanet=Lp,
                    Lstar=disk_arr[3], Rout=Rout, M_cpd=M_cpd, i=16
                )

                flux = out['F_nu_tot']
                tau_max = out['tau_nu_max']
            
                #print(flux)
                
                # Check if flux meets minimum threshold
                if flux >= 70 and chosen_M_cpd == 0.0:
                    # First time exceeding threshold - store it
                    chosen_flux = flux
                    chosen_M_cpd = M_cpd
                    prev_flux = flux
                    tau_max_chosen = tau_max
                    break  # Exit M_cpd loop
                

            # Store results
            Flux_vals[i, j] = chosen_flux
            M_cpd_chosen[i, j] = chosen_M_cpd
    
            
            # Print occasionally
            if i % 10 == 0 and j % 20 == 0:
                print(f"[{i},{j}] Rout/RH={frac:.3f}, Mp={Mp:.3f} -> Fnu={chosen_flux:.2f} µJy, M_cpd={chosen_M_cpd:.2e} Mjup")
                print(f" max Optical depth = {tau_max_chosen:.2e}")
                
    print("Number of non-zero flux values:", np.sum(Flux_vals > 0))
    print("Number of non-zero M_cpd values:", np.sum(M_cpd_chosen > 0))

    # plot histogram of flux distribution in log space
    plt.figure()
    plt.hist(np.log10(M_cpd_chosen[M_cpd_chosen > 0].flatten()), bins=30, color='blue', alpha=0.7)
    plt.xlabel(r'$M_{\rm CPD}$ [M$_{\rm Jup}$]')
    plt.ylabel('Count')
    plt.show()


    # Plot
    fig, ax = plt.subplots(figsize=(7, 5))

    cmap = 'Blues'  # default color map

    if plotmass:
        step = 0.25  # or 0.5 for coarser bins
        level_edges = np.arange(-5.5 - step/2, -3.0 + step/2, step)

        cf = ax.contourf(
            np.log10(Mp_grid), Rout_frac_grid, np.log10(M_cpd_chosen),
            levels=level_edges, cmap=cmap, extend='both', vmin=-5.5,  # Fix colorbar minimum
        vmax=-3.0 
        )  
        cbar = plt.colorbar(cf, ax=ax)
        cbar.set_label(r'$\log_{10}(M_{\rm dust}/M_{\rm Jup})$', fontsize=11)
    else:
        # Use linear flux levels for filled contours
        cf = ax.contourf(
            np.log10(Mp_grid), Rout_frac_grid, Flux_vals,
            levels=7, cmap=cmap, extend='both'
        )
        cbar = plt.colorbar(cf, ax=ax)
        cbar.set_label(r'$F_\nu\, [\mu{\rm Jy}]$', fontsize=11)

        

    # Overlay ALMA angular resolution limit
    #theta_beam = disk_arr[4]  # arcsec
    d_pc = disk_arr[2]
    # For each Mp, calculate the  Rout_frac = theta_beam * d_pc / RHill
    RHill_arr = rp * (Mp_grid / (3 * M_star_jup))**(1/3)
    Rout_frac_beam =  5/ (2*RHill_arr)  # this seems to give overestimation ,(theta_beam * d_pc)
    # for each planet mass, or for each Rout give a Mp 
    ax.plot(np.log10(Mp_grid), Rout_frac_beam, color='red', linestyle='-', linewidth=2, label='ALMA detection limit')


    # Axis labels
    ax.set_xlabel(r"$\log_{10}(M_p/M_{Jup})$")
    ax.set_ylabel(r"$R_{\rm cpd}/R_H$")
    ax.set_ylim(y_min, y_max)

    # Colorbar

    ax.set_title(f"{disk_arr[0]} at $r_p$={rp:.1f} AU, α={alpha}, κ=2.4 cm²/g")

    plt.tight_layout()
    plt.show()

In [ ]:
# Gap of HD143006 from Andrews paper
# D22:  103 (mas for r_peak), 
# D51  : 338 (mas for r_peak),

HD1423006_disk_arr = ["HD143006", 1.56, 167, 3.8]

D22_target_flux_arr = [71, 79, 88, 97, 111, 220]  # mu Jy for different recovery fractions from 0.5 to 1.0

D51_target_flux_arr = [85, 92, 98, 107, 120, 185]  # mu Jy for different recovery fractions from 0.5 to 1.0
rD22 = 22 #140 * HD1423006_disk_arr[2] / 1000  # arcsec to au
rD51 = 51 #315* HD1423006_disk_arr[2] / 1000  # arcsec to au

# Run for the D22 gap
Andrews_plots_flux_map(
    target_flux_arr=D22_target_flux_arr,
    disk_arr=HD1423006_disk_arr,
    rp=rD22,
    alpha=1e-3,
    plotmass=True
)

# Run for the D51 gap
Andrews_plots_flux_map(
    target_flux_arr=D51_target_flux_arr,
    disk_arr=HD1423006_disk_arr,
    rp=rD51,
    alpha=1e-3,
    plotmass=True
)

C:\Users\LHEM\AppData\Local\Temp\ipykernel_24092\2349032932.py:138: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  flux_integral = np.trapz(integrand, R)


[0,0] Rout/RH=0.050, Mp=0.010 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[0,20] Rout/RH=0.050, Mp=0.051 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[0,40] Rout/RH=0.050, Mp=0.260 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[0,60] Rout/RH=0.050, Mp=1.322 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[0,80] Rout/RH=0.050, Mp=6.734 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[10,0] Rout/RH=0.145, Mp=0.010 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[10,20] Rout/RH=0.145, Mp=0.051 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[10,40] Rout/RH=0.145, Mp=0.260 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[10,60] Rout/RH=0.145, Mp=1.322 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[10,80] Rout/RH=0.145, Mp=6.734 -> Fnu=0.00 µJy, M_cpd=0.00e+00 Mjup
 max Optical depth = 0.00e+00
[20,0] Rout/RH=0.

In [ ]:
print(80*60*100)

480000


In [ ]:
a = ((10**(-4)-10**(-6))/(3-0.4))
b = ((10**(-6)-(10**(-8))/(0.4-0.025)))
c = ((10**(-4)-(10**(-8))/(3-0.025)))

d = np.average([a,b,c])
